# 01 · Data ingestion and quality control

**Why.** Every later number depends on knowing exactly which file we analysed. This stage loads the survey table, checks its shape and codes against the codebook, and records hashes so anyone can confirm they have the same data.

**Data.** The three-class `diabetes_012_health_indicators_BRFSS2015.csv` from the [Kaggle Diabetes Health Indicators dataset](https://www.kaggle.com/datasets/alexteboul/diabetes-health-indicators-dataset), derived from the CDC's 2015 BRFSS survey: 253,680 adults, 21 self-reported answers and a diabetes status of 0 (none), 1 (prediabetes) or 2 (diabetes).

**Re-run it** (the raw and prepared files stay out of git):

```bash
diabetes-screening prepare --input data/raw/archive.zip \
  --output data/derived/combined_BRFSS2015.csv --endpoint combined
```

```bash
diabetes-screening run --input data/derived/combined_BRFSS2015.csv \
  --provenance data/derived/combined_BRFSS2015.provenance.json --output outputs/stage1-4/combined
```

In [1]:
# Every notebook reads the committed aggregate reports, so it runs from a fresh clone without the survey data.
from pathlib import Path
import json

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ENDPOINT = "combined"  # "prediabetes" shows the sensitivity analysis instead
REPORTS = ROOT / "reports"
STAGE_1_4 = REPORTS / ("uploaded-combined" if ENDPOINT == "combined" else "prediabetes")
MODELING = REPORTS / "modeling" / ENDPOINT
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", 100)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def figure(path, width=860):
    display(Image(filename=str(path), width=width))

## Where the data came from

`prepare` keeps the original three-class counts and maps them to the outcome explicitly. The hashes identify the exact files; the prepared file's hash is checked again by every later command.

In [2]:
provenance = read_json(STAGE_1_4 / "source_provenance.json")
pd.Series({
    "Source file": provenance["source_filename"],
    "Three-class CSV inside it": provenance["archive_member"],
    "CSV SHA-256": provenance["source_csv_sha256"][:16] + "…",
    "Prepared file SHA-256": provenance["derived_sha256"][:16] + "…",
    "Endpoint": provenance["endpoint"],
    "Label mapping (original code -> outcome)": provenance["label_mapping"],
    "Rows kept": f"{provenance['rows']:,} (excluded: {provenance['excluded_rows']:,})",
}).to_frame("value")

,value
Source file,archive.zip
Three-class CSV inside it,diabetes_012_health_indicators_BRFSS2015.csv
CSV SHA-256,a971809d0786d2d7…
Prepared file SHA-256,7e0e0421398c31ef…
Endpoint,combined
Label mapping (original code -> outcome),"{'0': 0, '1': 1, '2': 1}"
Rows kept,"253,680 (excluded: 0)"


In [3]:
names = {"0": "No diabetes", "1": "Prediabetes", "2": "Diabetes"}
original = pd.Series(provenance["original_target_counts"]).rename(index=names)
pd.DataFrame({"respondents": original, "share": original / original.sum()})

,respondents,share
No diabetes,213703,0.842
Prediabetes,4631,0.018
Diabetes,35346,0.139


The main analysis counts prediabetes **and** diabetes as positive (15.76%). The roadmap's "about 14%" is diabetes alone (35,346 / 253,680 = 13.93%), and Kaggle's separate binary file codes prediabetes as negative. The prediabetes-only sensitivity analysis drops the 35,346 people who already have diabetes (set `ENDPOINT = "prediabetes"` above).

## Shape, missing values and codes

In [4]:
quality = read_json(STAGE_1_4 / "quality.json")
print(f"{quality['rows']:,} rows x {quality['columns']} columns")
print(f"Missing values: {sum(quality['missing'].values())}; out-of-codebook values: {len(quality['invalid_values'])}")
print(f"Rows repeating an earlier respondent's exact answers: {quality['duplicate_rows']:,}")
pd.DataFrame(quality["ranges"]).T.astype(int)

253,680 rows x 22 columns
Missing values: 0; out-of-codebook values: 0
Rows repeating an earlier respondent's exact answers: 23,968


,min,max
Diabetes_binary,0,1
HighBP,0,1
HighChol,0,1
CholCheck,0,1
Smoker,0,1
Stroke,0,1
HeartDiseaseorAttack,0,1
PhysActivity,0,1
Fruits,0,1
Veggies,0,1


## The codebook

Every answer keeps its original survey code. Age, education, income and general health are ordered bands, not exact values.

In [5]:
pd.read_csv(STAGE_1_4 / "data_dictionary.csv")

,name,kind,minimum,maximum,description
0,Diabetes_binary,binary,0,1,0: no diabetes; 1: self-reported prediabetes or diabetes
1,HighBP,binary,0,1,"Ever told blood pressure was high; 0 no, 1 yes"
2,HighChol,binary,0,1,"Ever told cholesterol was high; 0 no, 1 yes"
3,CholCheck,binary,0,1,"Cholesterol checked in the past five years; 0 no, 1 yes"
4,BMI,numeric,12,98,Body mass index in kg/m²; source-derived observed bounds
5,Smoker,binary,0,1,"Smoked at least 100 cigarettes in lifetime; 0 no, 1 yes"
6,Stroke,binary,0,1,"Ever told of a stroke; 0 no, 1 yes"
7,HeartDiseaseorAttack,binary,0,1,"Coronary heart disease or myocardial infarction; 0 no, 1 yes"
8,PhysActivity,binary,0,1,"Physical activity in past 30 days, excluding job; 0 no, 1 yes"
9,Fruits,binary,0,1,"Consumes fruit at least once daily; 0 no, 1 yes"


**Takeaway.** The file has exactly the expected 253,680 rows and 22 columns, no missing or invalid codes, and its hashes match the team's original upload. About 24,000 rows repeat another respondent's answers; stage 2 decides what to do with them.

This is a research prototype for risk stratification, not a diagnostic or clinical decision tool. The outcome is self-reported, the data are cross-sectional and unweighted, and associations are not causes.